In [1]:
import libs_common
import rl_libs 

import numpy 

objc[10002]: Class SDLApplication is implemented in both /opt/homebrew/lib/python3.14/site-packages/envpool/gfootball/gfootball_envpool.so (0x10fb35f88) and /opt/homebrew/lib/python3.14/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x142f64890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[10002]: Class SDLAppDelegate is implemented in both /opt/homebrew/lib/python3.14/site-packages/envpool/gfootball/gfootball_envpool.so (0x10fb35fd8) and /opt/homebrew/lib/python3.14/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x142f648e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[10002]: Class SDLTranslatorResponder is implemented in both /opt/homebrew/lib/python3.14/site-packages/envpool/gfootball/gfootball_envpool.so (0x10fb36050) and /opt/homebrew/lib/python3.14/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x142f64958). This may cause spurio

# create environment wrapper

- 128 parallel envs
- env.step(actions), returns : states, rewards, dones

- actions.shape = (n_envs, ),   discrete actions, 4
- state.shape   = (n_envs, 8),   8 dims state space for each environment
- rewards.shape = (n_envs, ),   scalar reward value, float
- dones.shape = (n_envs, ),     scaler bool value if episode done

In [2]:
n_envs = 128
envs = libs_common.VecEnv(n_envs=n_envs)


In [3]:
rl_agent = rl_libs.A2CAgent(
    n_envs=n_envs,
    state_shape=(8,),
    n_actions=4,
    Model=rl_libs.MLPRLModel,
    learning_rate=0.002,
    n_steps=128,    
    gamma=0.99,
    entropy_beta=0.01,
    device="cpu",
)

state  = envs.reset()                             # (16, 8)
reward = numpy.zeros(n_envs, dtype=numpy.float32)
done   = numpy.zeros(n_envs, dtype=bool)

reward_episode_sum = numpy.zeros(n_envs) 
reward_episode     = 0.0
episode_count      = 0

for n in range(200000):

    actions = rl_agent.step(state, reward, done, True)

    state, reward, done = envs.step(actions)

    reward_episode_sum += reward

    # several envs may finish on the same step
    for i in numpy.flatnonzero(done):
        reward_episode = 0.9 * reward_episode + 0.1 * reward_episode_sum[i]
        reward_episode_sum[i] = 0.0
        episode_count += 1
        
    if (n % 1000) == 0:
        print(n, episode_count, round(float(reward_episode), 4))


MLPRLModel(
  (lin0): Linear(in_features=8, out_features=128, bias=True)
  (act0): SiLU()
  (lin1): Linear(in_features=128, out_features=128, bias=True)
  (act1): SiLU()
  (actor): Linear(in_features=128, out_features=4, bias=True)
  (critic): Linear(in_features=128, out_features=1, bias=True)
)
n_envs       =  128
n_steps      =  128
gamma        =  0.99
entropy_beta =  0.01
device       =  cpu
0 0 0.0
1000 1325 -1.821
2000 2668 -1.5554
3000 3930 -2.3088
4000 5262 -1.915
5000 6688 -1.6336
6000 8126 -1.7109
7000 9532 -1.3165
8000 10859 -1.2523
9000 12144 -1.2778
10000 13400 -1.4509
11000 14528 -1.1938
12000 15526 -1.0329
13000 16451 -0.9625
14000 17350 -0.5716
15000 18121 -0.2927
16000 18798 -0.139
17000 19271 -0.276
18000 19634 -0.521
19000 19876 -0.4429
20000 20133 -0.4357
21000 20421 -0.1318
22000 20684 0.0019
23000 20905 -0.2654
24000 21116 -0.023
25000 21293 -0.2114
26000 21441 -0.3408
27000 21618 -0.2723
28000 21768 -0.2935
29000 21914 -0.158
30000 22062 -0.1322
31000 22225 0.156